# Transformer Architecture
### Zooming out from the two towers to the whole machine — every component, one forward pass

---

## 📖 The Story

Topic 5 wired the encoder and decoder towers together. But a Transformer is far more than attention — about two-thirds of every block's parameters live in a feed-forward network, and the model only trains because of residual connections, layer normalization, and careful embedding scaling.

This notebook assembles the *complete* machine from scratch and uses it for a real task: **next-token prediction**. Token IDs go in one end; a probability distribution over the vocabulary comes out the other. We build every non-attention part — embeddings with √d_model scaling, the position-wise FFN, the Add & Norm sub-layer wrapper, the N-block stack, and a weight-tied output head — and watch a representation flow all the way through.

---

**What this notebook covers:**
- Building the full pipeline: embedding × √d_model → positional encoding → N blocks → output head
- Implementing the position-wise feed-forward network and the Add & Norm sub-layer wrapper
- Stacking N identical decoder-style blocks with a causal mask (a GPT-style language model)
- Predicting the next token, and breaking down where the parameters actually live
- Verifying the block against PyTorch's `nn.TransformerEncoder`

**Prerequisites:**
- Self-Attention (Topic 2), Multi-Head Attention (Topic 3), Positional Encoding (Topic 4), Encoder-Decoder (Topic 5)
- NumPy fundamentals

**Note:** We build a small decoder-only (causal) stack — the cleanest 'complete Transformer doing a task'. An encoder-only model (BERT) is the same block without the mask; an encoder-decoder (Topic 5) adds cross-attention. The block is shared.

In [ ]:
# --- All Imports ---
import numpy as np                        # Core math — full Transformer from scratch
import matplotlib.pyplot as plt           # Plotting
import seaborn as sns                     # Heatmaps
import torch                              # PyTorch — verification
import torch.nn as nn
import torch.nn.functional as F
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print('All imports successful ✅')

## Part 1: Theory Recap

Five things to know before we write any code:

- **The model is one block repeated N times**: every Transformer block has the identical shape — attention, then a feed-forward network, each wrapped in Add & Norm.
- **Embeddings are scaled by √d_model**: token vectors are multiplied by √d_model so they are not drowned out by the positional encodings added on top.
- **The FFN is where most parameters live**: a position-wise two-layer MLP (`d_model → d_ff → d_model`) applied identically to every token — roughly two-thirds of a block's weights.
- **Add & Norm = residual + LayerNorm**: `LayerNorm(x + Sublayer(x))` gives gradients a highway around each sub-layer, making deep stacks trainable.
- **The output head is often weight-tied**: the final projection to vocabulary logits reuses the embedding matrix (`X · Wₑᵀ`), saving parameters and linking input/output token spaces.

## Setting Up a Real Example Sentence

We define a tiny vocabulary and feed the model the prompt **"the cat sat on the"** as token IDs. Because this is a causal language model, the job is to predict the token that comes *next*.

Weights are random (an untrained model), so the prediction itself is meaningless — the point is to watch a real set of token IDs flow through every component and come out as a vocabulary distribution.

In [ ]:
# A tiny vocabulary and an input prompt (as token IDs)
vocab = ['<pad>', 'the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran']
stoi  = {w: i for i, w in enumerate(vocab)}
prompt = ['the', 'cat', 'sat', 'on', 'the']
ids = [stoi[w] for w in prompt]

V       = len(vocab)   # vocabulary size
d_model = 16           # embedding / model dimension
n_heads = 4            # attention heads  (d_k = 4)
d_ff    = 64           # FFN hidden size (4 x d_model, as in the original paper)
N       = 2            # number of stacked blocks
L       = len(ids)

print(f'prompt : {prompt}')
print(f'ids    : {ids}')
print(f'V={V}, d_model={d_model}, n_heads={n_heads}, d_ff={d_ff}, N={N}, L={L}')

## Part 2: Building the Full Transformer From Scratch

We assemble the complete machine in order:
1. Building blocks: `softmax`, `layer_norm`, `ffn`, `multi_head_attention`, `positional_encoding`
2. One **block**: attention → Add & Norm → FFN → Add & Norm
3. The **full forward pass**: embed × √d_model → + positional encoding → N blocks → weight-tied head → softmax

In [ ]:
def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True)); return e / e.sum(axis=axis, keepdims=True)

def layer_norm(x, gamma, beta, eps=1e-5):
    mu = x.mean(-1, keepdims=True); var = x.var(-1, keepdims=True)
    return gamma * (x - mu) / np.sqrt(var + eps) + beta

def ffn(x, W1, b1, W2, b2):
    """Position-wise feed-forward: ReLU(xW1+b1)W2+b2 — applied identically to every token."""
    return np.maximum(0, x @ W1 + b1) @ W2 + b2

def multi_head_attention(X, WQ, WK, WV, WO, n_heads, mask=None):
    L, dm = X.shape; dk = dm // n_heads
    Q = (X @ WQ).reshape(L, n_heads, dk).transpose(1, 0, 2)
    K = (X @ WK).reshape(L, n_heads, dk).transpose(1, 0, 2)
    Vv = (X @ WV).reshape(L, n_heads, dk).transpose(1, 0, 2)
    scores = np.matmul(Q, K.transpose(0, 2, 1)) / np.sqrt(dk)
    if mask is not None: scores = scores + mask
    w = softmax(scores, axis=-1)
    ctx = np.matmul(w, Vv).transpose(1, 0, 2).reshape(L, dm)
    return ctx @ WO, w

def positional_encoding(L, dm):
    pe = np.zeros((L, dm)); pos = np.arange(L)[:, None]
    div = np.exp(np.arange(0, dm, 2) * (-np.log(10000.0) / dm))
    pe[:, 0::2] = np.sin(pos * div); pe[:, 1::2] = np.cos(pos * div)
    return pe

print('Building blocks ready ✅')

In [ ]:
# --- Parameter initialization for the whole model ('pretend trained') ---
rand = lambda *s: np.random.randn(*s) * 0.3
Emb = rand(V, d_model)                                   # token embedding table (also the tied head)
def attn_params(): return tuple(rand(d_model, d_model) for _ in range(4))   # WQ,WK,WV,WO
def ffn_params():  return rand(d_model, d_ff), np.zeros(d_ff), rand(d_ff, d_model), np.zeros(d_model)
def ln_params():   return np.ones(d_model), np.zeros(d_model)               # gamma, beta

blocks = [{'attn': attn_params(), 'ffn': ffn_params(),
           'ln1': ln_params(), 'ln2': ln_params()} for _ in range(N)]
causal_mask = np.triu(np.full((L, L), -1e9), k=1)        # GPT-style: no peeking ahead

def transformer_block(X, blk, n_heads, mask):
    """One block: Attention -> Add&Norm -> FFN -> Add&Norm (post-norm)."""
    a, w = multi_head_attention(X, *blk['attn'], n_heads, mask=mask)
    X = layer_norm(X + a, *blk['ln1'])          # Add & Norm  #1
    f = ffn(X, *blk['ffn'])
    X = layer_norm(X + f, *blk['ln2'])          # Add & Norm  #2
    return X, w

# ---------- FULL FORWARD PASS ----------
X = Emb[ids] * np.sqrt(d_model)                 # 1) embed + SCALE by sqrt(d_model)
X = X + positional_encoding(L, d_model)         # 2) add positional encoding (Topic 4)
reps = [X.copy()]                               # keep the representation after each stage
for b, blk in enumerate(blocks):
    X, _ = transformer_block(X, blk, n_heads, causal_mask)   # 3) N stacked blocks
    reps.append(X.copy())
logits = X @ Emb.T                              # 4) weight-tied output head (X . Wₑᵀ)
probs  = softmax(logits[-1])                    # 5) next-token distribution at the LAST position

pred = vocab[int(np.argmax(probs))]
print(f'Input prompt : {prompt}')
print(f'Logits shape : {logits.shape}  (one row per position, {V} vocab scores each)')
print(f'Predicted next token (untrained, random): \'{pred}\'')
print(f'Next-token probabilities: {dict(zip(vocab, np.round(probs,3)))}')

## What Just Happened?

We ran a complete Transformer language model end to end:

- **Embedding × √d_model** turned token IDs into vectors and scaled them so positional encodings (added next) would not dominate.
- Each of the **N blocks** mixed context with attention, then transformed each token with the **FFN** — every sub-layer wrapped in **Add & Norm**.
- After the final block, the **weight-tied head** (`X · Wₑᵀ`) projected each token's vector back into `V` vocabulary scores.
- A **softmax** over the last position gave a probability distribution for the next token.

This is the entire architecture. BERT is the same block without the causal mask; the original Transformer (Topic 5) adds a second tower and cross-attention. Everything downstream — pretraining, fine-tuning, modern LLMs — is this machine, scaled.

In [ ]:
# --- Visualisation 1: Where do the parameters actually live? ---
# Use realistic original-Transformer-base sizes to show the true proportions.
DM, DFF, NB, VV = 512, 2048, 6, 30000
p_embed = VV * DM                              # token embedding (tied with head -> counted once)
p_attn  = NB * 4 * DM * DM                     # WQ,WK,WV,WO per block
p_ffn   = NB * (DM * DFF + DFF * DM)           # two linear layers per block
p_ln    = NB * 2 * 2 * DM                      # gamma+beta for two LayerNorms per block

labels = ['Embedding\n(tied head)', 'Attention\n(4 matrices)', 'Feed-Forward\n(FFN)', 'LayerNorm']
vals   = [p_embed, p_attn, p_ffn, p_ln]
colors = ['#8172B3', '#4C72B0', '#DD8452', '#55A868']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(labels, np.array(vals)/1e6, color=colors)
axes[0].set_title('Parameters by component (Transformer base: d_model=512, N=6)', fontweight='bold')
axes[0].set_ylabel('Parameters (millions)')
for i, v in enumerate(vals): axes[0].text(i, v/1e6, f'{v/1e6:.1f}M', ha='center', va='bottom', fontweight='bold')

block_only = [p_attn, p_ffn, p_ln]
axes[1].pie(block_only, labels=['Attention','FFN','LayerNorm'], autopct='%1.0f%%',
            colors=['#4C72B0','#DD8452','#55A868'], startangle=90)
axes[1].set_title('Inside the blocks: FFN dominates', fontweight='bold')

plt.tight_layout(); plt.show()
print(f'FFN is {p_ffn/(p_attn+p_ffn+p_ln)*100:.0f}% of all in-block parameters — attention gets the fame, the FFN holds the weight.')

In [ ]:
# --- Visualisation 2: How the representation evolves through the stack ---
# Track the average vector norm at each stage — residual connections make it grow steadily.
stage_names = ['embed+pos'] + [f'after block {i+1}' for i in range(N)]
avg_norms = [np.linalg.norm(r, axis=-1).mean() for r in reps]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(stage_names, avg_norms, marker='o', linewidth=2, color='steelblue')
axes[0].set_title('Average representation norm through the stack\n(LayerNorm keeps it controlled)', fontweight='bold')
axes[0].set_ylabel('mean L2 norm per token'); axes[0].grid(True, alpha=0.3)

# How much each block CHANGES the representation (residual update size)
deltas = [np.linalg.norm(reps[i+1]-reps[i], axis=-1).mean() for i in range(N)]
axes[1].bar([f'block {i+1}' for i in range(N)], deltas, color='#DD8452')
axes[1].set_title('How much each block updates the residual stream', fontweight='bold')
axes[1].set_ylabel('mean change per token'); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout(); plt.show()
print('Each block ADDS a residual update on top of the previous representation —')
print('information accumulates in the \'residual stream\' rather than being overwritten.')

## Part 3: PyTorch Verification

Two checks:
1. **Numeric** — our causal multi-head attention (the heart of the block) should match PyTorch's `F.scaled_dot_product_attention(..., is_causal=True)`.
2. **End-to-end** — a real `nn.TransformerEncoder` with a causal mask should accept our input and return one vector per position, exactly like our stack.

In [ ]:
# --- Check 1: causal multi-head attention vs PyTorch SDPA (block 1, head outputs) ---
WQ, WK, WV, WO = blocks[0]['attn']; dk = d_model // n_heads
X0 = reps[0]
Qh = (X0 @ WQ).reshape(L, n_heads, dk).transpose(1, 0, 2)
Kh = (X0 @ WK).reshape(L, n_heads, dk).transpose(1, 0, 2)
Vh = (X0 @ WV).reshape(L, n_heads, dk).transpose(1, 0, 2)
_, w0 = multi_head_attention(X0, *blocks[0]['attn'], n_heads, mask=causal_mask)
our_ctx = np.matmul(w0, Vh)
t = lambda z: torch.tensor(z, dtype=torch.float32)
with torch.no_grad():
    ref_ctx = F.scaled_dot_product_attention(t(Qh), t(Kh), t(Vh), is_causal=True).numpy()
max_diff = np.max(np.abs(our_ctx - ref_ctx))
print('=== Check 1: Causal MHA vs PyTorch SDPA ===')
print(f'  Max absolute difference: {max_diff:.2e}')
print('  ✅ Match' if max_diff < 1e-4 else '  ❌ Mismatch')

# --- Check 2: nn.TransformerEncoder end-to-end ---
print('\n=== Check 2: torch.nn.TransformerEncoder ===')
layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
                                   batch_first=True)
encoder = nn.TransformerEncoder(layer, num_layers=N)
cm = nn.Transformer.generate_square_subsequent_mask(L)
with torch.no_grad():
    out = encoder(torch.randn(1, L, d_model), mask=cm)
print(f'  input (1, {L}, {d_model})  ->  output {tuple(out.shape)}')
print('  ✅ Same shape as our scratch stack: one context vector per position.')

## Part 4: Hyperparameter Experiments

Two experiments connecting our tiny model to real ones:
1. **Scaling with depth (N) and width (d_model)** — how the parameter count grows.
2. **Recreating real configs** — base, large, and a GPT-3-scale model, all from the same formula.

In [ ]:
# Parameter formula for a decoder-only stack (tied head, ignoring biases)
def total_params(dm, dff, nb, vocab):
    embed = vocab * dm
    per_block = 4*dm*dm + 2*dm*dff + 4*dm   # attn + ffn + 2 layernorms
    return embed + nb * per_block

# Experiment 1: params vs depth, for a few widths
N_vals = [2, 4, 6, 12, 24]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for dm, dff in [(256,1024), (512,2048), (768,3072)]:
    ys = [total_params(dm, dff, n, 30000)/1e6 for n in N_vals]
    axes[0].plot(N_vals, ys, marker='o', linewidth=2, label=f'd_model={dm}')
axes[0].set_title('Parameters vs depth (N)', fontweight='bold')
axes[0].set_xlabel('Number of blocks (N)'); axes[0].set_ylabel('Parameters (millions)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Experiment 2: recreate named configs
configs = {'Base':(512,2048,6), 'Large':(1024,4096,24), 'GPT-3 175B*':(12288,49152,96)}
names = list(configs); sizes = [total_params(*configs[c], 50000)/1e9 for c in names]
axes[1].bar(names, sizes, color=['#55A868','#4C72B0','#C44E52'])
axes[1].set_title('Same formula, three scales', fontweight='bold')
axes[1].set_ylabel('Parameters (billions)')
for i, v in enumerate(sizes): axes[1].text(i, v, f'{v:.1f}B', ha='center', va='bottom', fontweight='bold')
plt.tight_layout(); plt.show()
print('* GPT-3 estimate from architecture dims only — the SAME block, just stacked wider and deeper.')

## Common Mistakes Students Make

**Mistake 1: Forgetting to scale embeddings by √d_model.**
Without the scaling, raw embeddings are small relative to the positional encodings (which range over [-1, 1] across many dimensions), so position can swamp meaning at the input. Multiplying by √d_model keeps the two signals comparable.

**Mistake 2: Thinking attention is the bulk of the model.**
In every block the position-wise FFN holds about two-thirds of the parameters. Attention routes information; the FFN stores and transforms it. A mental model that ignores the FFN misjudges where a Transformer's capacity actually comes from.

**Mistake 3: Confusing the three Transformer families as different architectures.**
Encoder-only (BERT), decoder-only (GPT), and encoder-decoder (Topic 5) are built from the *same* block. The differences are masking (causal or not) and whether cross-attention is present — not a different architecture.

## Part 5: Interview Corner

**Question: Walk me through a Transformer's complete forward pass, from token IDs to a next-token probability, naming every component.**

A complete answer:

**1. Embedding + scaling.** Each token ID indexes a learned embedding table, and the resulting vectors are multiplied by √d_model so they are not dominated by the positional signal added next.

**2. Positional encoding.** A fixed (or learned) position pattern is added so the otherwise order-blind model knows token order (Topic 4).

**3. N identical blocks.** Each block runs multi-head attention (mixing information across tokens), wrapped in Add & Norm, then a position-wise feed-forward network (transforming each token independently), again wrapped in Add & Norm. The residual connections let gradients flow straight through the stack; LayerNorm keeps activations stable.

**4. Output head.** The final representation is projected to vocabulary logits — often by reusing the embedding matrix (weight tying) — and a softmax turns those logits into a next-token probability distribution.

**The key insight:** the architecture is *one block repeated N times*, bracketed by an input stage (embedding + positions) and an output stage (head + softmax). Attention gets the headlines, but the FFN holds most of the parameters, and residual + LayerNorm are what make stacking dozens of blocks trainable at all.

## Key Takeaways

Five bullets you must remember for placement interviews:

- **A Transformer is one block repeated N times**, bracketed by an input stage (embedding × √d_model + positional encoding) and an output stage (linear head + softmax).
- **The FFN holds most of the parameters**: a position-wise two-layer MLP is ~2/3 of each block — attention routes information, the FFN stores and transforms it.
- **Add & Norm makes depth trainable**: `LayerNorm(x + Sublayer(x))` gives gradients a residual highway and keeps activations stable across deep stacks.
- **Embedding scaling and weight tying are easy to miss but matter**: √d_model keeps input signals balanced; tying the head to the embedding saves parameters and links token spaces.
- **The three families share this block**: BERT (no mask), GPT (causal mask), and encoder-decoder (Topic 5, + cross-attention) are the same architecture, configured differently.